# Laboratorio 1: datos estructurados con SQL

**Objetivo:** reconocer entidades, claves y relaciones usando un conjunto sintetico de cursos y estudiantes. El ejemplo se ejecuta en memoria con Python estandar; luego contrasta el SQL que usa PostgreSQL en la API.

In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute("PRAGMA foreign_keys = ON")
connection.executescript("""
CREATE TABLE courses (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE,
    area TEXT NOT NULL
);
CREATE TABLE students (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    course_id INTEGER NOT NULL REFERENCES courses(id)
);
""")
connection.executemany(
    "INSERT INTO courses (name, area) VALUES (?, ?)",
    [("Mineria de datos", "Analitica"), ("Bases de datos", "Computacion")],
)
connection.executemany(
    "INSERT INTO students (name, email, course_id) VALUES (?, ?, ?)",
    [("Ana Ruiz", "ana@example.edu", 1), ("Luis Mora", "luis@example.edu", 1), ("Eva Gil", "eva@example.edu", 2)],
)
connection.commit()
print("Tablas creadas y datos sinteticos insertados")

## Consulta relacional: estudiantes y cursos

El `JOIN` expresa una relacion declarada mediante una clave foranea. En PostgreSQL, las tablas y restricciones equivalentes estan en `scripts/schema.sql`.

In [ ]:
query = """
SELECT c.name AS course, s.name AS student, s.email
FROM students AS s
JOIN courses AS c ON c.id = s.course_id
ORDER BY c.name, s.name
"""
rows = connection.execute(query).fetchall()
for row in rows:
    print(row)

assert len(rows) == 3
assert rows[0][0] == "Bases de datos"

## Agregacion y restriccion

Agrupamos estudiantes por curso y luego comprobamos que la base rechaza un correo repetido. En SQL real, el `UNIQUE` evita duplicados incluso si el alta no pasa por esta API.

In [ ]:
summary = connection.execute("""
SELECT c.name, COUNT(s.id) AS student_count
FROM courses AS c
LEFT JOIN students AS s ON s.course_id = c.id
GROUP BY c.id, c.name
ORDER BY c.name
""").fetchall()
summary

## Reto

1. Agrega una columna `semester` a cursos y decide si debe ser obligatoria.
2. Intenta insertar dos cursos con el mismo nombre y observa el error.
3. Explica que garantiza la clave foranea y que no garantiza una referencia escrita en un documento MongoDB.
4. Compara este ejemplo SQLite en memoria con el esquema PostgreSQL del proyecto.

**Cierre:** el modelo relacional es util cuando las entidades, relaciones y reglas de integridad son parte central del problema. Los datos de este ejemplo son sinteticos.